# B. Train + evaluate (GPU)

**Settings:** Accelerator **GPU T4** (T4 has tensor cores → AMP/fp16 is fast; P100 does not) · Internet **On**
(git clone + ImageNet weights) · **Add Input** → your private dataset `speedplus-960`.

Workflow: run cells 1–5 interactively (tests + a 1-epoch tiny run, a few minutes) → then for the real runs
use **Save Version → Save & Run All** so the run survives closing the browser (12 h limit per session,
~30 GPU h/week). Results land in `/kaggle/working/repo/outputs/runs/<name>_<time>/` (metrics.csv, best.pt, last.pt).

In [ ]:
# 1. Find the dataset (Kaggle may keep it as a zip or auto-extract it) -> local scratch copy
import glob, os
zips = glob.glob("/kaggle/input/**/speedplus_960.zip", recursive=True)
cams = glob.glob("/kaggle/input/**/camera.json", recursive=True)
ROOT = "/kaggle/temp/sp960"
if zips:
    if not os.path.exists(f"{ROOT}/camera.json"):
        !mkdir -p {ROOT} && unzip -q {zips[0]} -d {ROOT}
elif cams:
    ROOT = os.path.dirname(cams[0])
else:
    raise FileNotFoundError("attach the speedplus-960 dataset (Add Input)")
print("ROOT =", ROOT)
!ls {ROOT} && cat {ROOT}/camera.json | head -20

In [ ]:
# 2. Code at a known commit (the hash is also logged into each run's config.yaml)
%cd /kaggle/working
!rm -rf repo && git clone -q https://github.com/manauwar713/speedplus-pose-estimation.git repo
%cd /kaggle/working/repo
!git log --oneline -1
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. Unit tests on the cloud machine (different numpy/OpenCV/torch versions than the laptop!)
!python -m pytest -q tests

In [ ]:
# 4. GT pipeline check on the REAL resized data (exact ~1e-6 deg, heatmap ~0.02 deg expected)
!python scripts/check_pipeline_gt.py --root {ROOT} --max 300

In [ ]:
# 5. Plumbing run: 1 epoch on 2k images. Look at: loss falling from ~3.1e-3, s/epoch, GPU util.
!python -m src.train --config configs/baseline_r50.yaml --set data.root={ROOT} name=plumb data.train_max_samples=2000 data.val_max_samples=200 train.epochs=1 train.val_every=1 train.log_every=20

### Your experiments (you decide what to run and read the curves)
Debug run first (CLAUDE.md: 10k subset), then the full baseline. Edit the `--set` list; every value is logged.

In [ ]:
# 6. Debug run: 10k images, 10 epochs
!python -m src.train --config configs/baseline_r50.yaml --set data.root={ROOT} name=r50_10k data.train_max_samples=10000 train.epochs=10

In [ ]:
# 7. Full baseline (all 47,966 train images, 40 epochs) -- run in a committed version
# !python -m src.train --config configs/baseline_r50.yaml --set data.root={ROOT}

In [ ]:
# 8. Evaluate the best checkpoint on all three domains (full validation set too)
run = sorted(glob.glob("outputs/runs/*/best.pt"))[-1]
print(run)
!python -m src.evaluate --config configs/baseline_r50.yaml --ckpt {run} --set data.root={ROOT} --splits validation lightbox sunlamp --out {os.path.dirname(run)}/eval.json

In [ ]:
# 9. Curves from metrics.csv
import pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv(sorted(glob.glob("outputs/runs/*/metrics.csv"))[-1])
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].semilogy(df.epoch, df.train_loss, label="train"); ax[0].semilogy(df.epoch, df.val_loss, "o-", label="val"); ax[0].legend(); ax[0].set_title("heatmap loss")
ax[1].plot(df.epoch, df.mean_e_q_deg, "o-", label="mean"); ax[1].plot(df.epoch, df.median_e_q_deg, "o-", label="median"); ax[1].legend(); ax[1].set_title("val E_q [deg]")
ax[2].plot(df.epoch, df.mean_e_spec, "o-"); ax[2].set_title("val E*_pose")
plt.show()